# Coffee Sales Analysis

## Import libraries and load CSV file

In [11]:
import pandas as pd
import numpy as np
df = pd.read_csv('coffee_shop_sales.csv', sep=';')

## 1. Data Quality Check

Before performing the analysis, I checked the dataset structure,
data types, missing values and duplicate records.

In [12]:
df.shape

(149116, 11)

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 149116 entries, 0 to 149115
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_id    149116 non-null  int64  
 1   transaction_date  149116 non-null  str    
 2   transaction_time  149116 non-null  str    
 3   transaction_qty   149116 non-null  int64  
 4   store_id          149116 non-null  int64  
 5   store_location    149116 non-null  str    
 6   product_id        149116 non-null  int64  
 7   unit_price        149116 non-null  float64
 8   product_category  149116 non-null  str    
 9   product_type      149116 non-null  str    
 10  product_detail    149116 non-null  str    
dtypes: float64(1), int64(4), str(6)
memory usage: 12.5 MB


In [14]:
df.isna().sum()

transaction_id      0
transaction_date    0
transaction_time    0
transaction_qty     0
store_id            0
store_location      0
product_id          0
unit_price          0
product_category    0
product_type        0
product_detail      0
dtype: int64

In [15]:
df.duplicated().sum()

np.int64(0)

In [21]:
df['transaction_id'].nunique()

149116

In [16]:
df.head()

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,01.01.2023,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,01.01.2023,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,01.01.2023,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,01.01.2023,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,01.01.2023,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


The dataset contains 149,116 records across 11 columns. No missing values or duplicate rows were identified. However, transaction_date and transaction_time are currently stored as strings and should be converted to appropriate datetime formats before performing time-based analysis.

## 2. Data Preparation

Converting data to the required type

In [30]:
df['transaction_date'] = pd.to_datetime(
    df['transaction_date'], 
    dayfirst=True
    )
df['transaction_time'] = pd.to_datetime(
    df['transaction_time'],
    format='%H:%M:%S'
    )
df['hour'] = df['transaction_time'].dt.hour

In [31]:
df['transaction_date'].head()

0   2023-01-01
1   2023-01-01
2   2023-01-01
3   2023-01-01
4   2023-01-01
Name: transaction_date, dtype: datetime64[us]

In [32]:
df['transaction_time'].head()

0   1900-01-01 07:06:11
1   1900-01-01 07:08:56
2   1900-01-01 07:14:04
3   1900-01-01 07:20:24
4   1900-01-01 07:22:41
Name: transaction_time, dtype: datetime64[us]

In [37]:
df['revenue'] = df['transaction_qty'] * df['unit_price']

The date and time fields were converted to appropriate datetime formats. An `hour` feature was extracted from `transaction_time` to support
time-based analysis. Calculated transaction revenue as `transaction_qty × unit_price`.

## 3. Data Validation

Verification of conversion correctness

In [33]:
df.dtypes

transaction_id               int64
transaction_date    datetime64[us]
transaction_time    datetime64[us]
transaction_qty              int64
store_id                     int64
store_location                 str
product_id                   int64
unit_price                 float64
product_category               str
product_type                   str
product_detail                 str
hour                         int32
dtype: object

In [34]:
df['hour'].min(), df['hour'].max()

(np.int32(6), np.int32(20))

In [35]:
df['hour'].value_counts().sort_index()

hour
6      4594
7     13428
8     17654
9     17764
10    18545
11     9766
12     8708
13     8714
14     8933
15     8979
16     9093
17     8745
18     7498
19     6092
20      603
Name: count, dtype: int64

The data was successfully transformed and validated. All required fields are in the correct format and ready for further analysis.

## 4. Exploratory Data Analysis

Analysis of distribution, variability, and unusual values

### 4.1. Distribution of transaction revenue

In [38]:
df['revenue'].describe()

count    149116.000000
mean          4.686367
std           4.227099
min           0.800000
25%           3.000000
50%           3.750000
75%           6.000000
max         360.000000
Name: revenue, dtype: float64

In [42]:
df['revenue'].median()

np.float64(3.75)

Transaction revenue is right-skewed (as the mean is higher than the median) driven by high-value transactions.
50% of transactions generate between \$3.00 and \$6.00 in revenue. The standard deviation of \$4.23 indicates noticeable variability in transaction revenue.
The large gap between the typical transaction revenue and the maximum value of \$360 suggests the presence of high-value transactions and potential outliers.

### 4.2. Outlier Analysis

In [44]:
Q1 = df['revenue'].quantile(0.25)
Q3 = df['revenue'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df['revenue'] < lower_bound) |
    (df['revenue'] > upper_bound)
]

len(outliers)

3273

In [45]:
outliers['revenue'].describe()

count    3273.000000
mean       19.382215
std        20.582864
min        10.950000
25%        12.750000
50%        15.000000
75%        21.000000
max       360.000000
Name: revenue, dtype: float64

In [46]:
df.nlargest(10, 'revenue')[[
    'transaction_id',
    'transaction_date',
    'transaction_time',
    'store_location',
    'product_category',
    'product_type',
    'revenue'
]]

,transaction_id,transaction_date,transaction_time,store_location,product_category,product_type,revenue
9310,9340,2023-01-17,1900-01-01 09:05:20,Hell's Kitchen,Coffee beans,Premium Beans,360.0
9365,9395,2023-01-17,1900-01-01 09:55:47,Hell's Kitchen,Coffee beans,Premium Beans,360.0
68806,68976,2023-04-17,1900-01-01 09:55:47,Hell's Kitchen,Coffee beans,Premium Beans,360.0
68981,69151,2023-04-17,1900-01-01 11:18:31,Hell's Kitchen,Coffee beans,Premium Beans,360.0
97979,98233,2023-05-17,1900-01-01 09:05:20,Hell's Kitchen,Coffee beans,Premium Beans,360.0
98275,98529,2023-05-17,1900-01-01 11:18:31,Hell's Kitchen,Coffee beans,Premium Beans,360.0
133186,133523,2023-06-17,1900-01-01 09:55:47,Hell's Kitchen,Coffee beans,Premium Beans,360.0
133337,133674,2023-06-17,1900-01-01 10:41:11,Hell's Kitchen,Coffee beans,Premium Beans,360.0
133407,133744,2023-06-17,1900-01-01 11:18:31,Hell's Kitchen,Coffee beans,Premium Beans,360.0
148702,149043,2023-06-30,1900-01-01 11:18:31,Hell's Kitchen,Coffee beans,Premium Beans,360.0


In [52]:
outlier_revenue_share = (
    outliers['revenue'].sum() / df['revenue'].sum() * 100
)

outlier_revenue_share

np.float64(9.077972336292348)

The IQR method identified 3,273 outlier transactions, representing approximately 2.2% of all transactions.
Outlier revenue ranges from \$10.95 to \$360, with a median of \$15.00, indicating that most outliers are moderately higher-value transactions rather than extreme values.
The highest-value transactions (\$360) are associated with Premium Beans at the Hell's Kitchen store and occur repeatedly throughout the analyzed period.
The repeated occurrence of these $360 transactions suggests that they are likely valid high-value purchases rather than data errors.
Outlier transactions represent only 2.2% of all transactions but account for 9.1% of total revenue, indicating that high-value transactions have a noticeable impact on overall sales.

### 4.3. Items per Transaction

In [49]:
items_per_transaction = (
    df.groupby('transaction_id')['transaction_qty']
      .sum()
)

In [50]:
items_per_transaction.describe()

count    149116.000000
mean          1.438276
std           0.542509
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max           8.000000
Name: transaction_qty, dtype: float64

In [51]:
items_per_transaction.median()

np.float64(1.0)

The dataset contains an average of 1.44 items per transaction.
The median transaction contains 1 item, while 75% of transactions contain no more than 2 items.
The number of items per transaction ranges from 1 to 8, indicating that most purchases consist of a small number of items.
The difference between the mean (1.44) and median (1) suggests that transactions with multiple items slightly increase the average basket size.

### 4.4. Transaction Revenue vs. Number of Items

In [ ]:
transaction_summary = (
    df.groupby('transaction_id')
      .agg(
          revenue=('revenue', 'sum'),
          items=('transaction_qty', 'sum')
      )
)

In [59]:
transaction_summary['revenue'].corr(
    transaction_summary['items']
)

np.float64(0.3562308526230813)

In [60]:
revenue_by_items = (
    transaction_summary
    .groupby('items')['revenue']
    .agg(
        transactions='count',
        total_revenue='sum',
        avg_revenue='mean',
        median_revenue='median'
    )
)

revenue_by_items

,transactions,total_revenue,avg_revenue,median_revenue
items,,,,
1,87159,322430.83,3.699341,3.1
2,58642,343529.60,5.858081,6.0
3,3279,28829.10,8.792040,9.0
4,23,206.80,8.991304,3.2
6,3,216.00,72.000000,72.0
8,10,3600.00,360.000000,360.0


In [61]:
transaction_summary['revenue_per_item'] = (
    transaction_summary['revenue'] /
    transaction_summary['items']
)

transaction_summary['revenue_per_item'].describe()

count    149116.000000
mean          3.382219
std           2.658723
min           0.800000
25%           2.500000
50%           3.000000
75%           3.750000
max          45.000000
Name: revenue_per_item, dtype: float64

In [62]:
revenue_per_item_by_items = (
    transaction_summary
    .groupby('items')['revenue_per_item']
    .mean()
)

revenue_per_item_by_items

items
1     3.699341
2     2.929041
3     2.930680
4     2.247826
6    12.000000
8    45.000000
Name: revenue_per_item, dtype: float64

The number of items per transaction has a moderate positive relationship with transaction revenue (correlation = 0.36), indicating that larger transactions generally tend to generate higher revenue.
Average transaction revenue increases from \$3.70 for one-item transactions to \$5.86 for two-item transactions and \$8.79 for three-item transactions.
Average revenue per item decreases from \$3.70 for one-item transactions to approximately \$2.93 for transactions with two or three items, suggesting that larger baskets tend to have a lower average revenue per item.
Transactions with 4+ items are extremely rare, so their revenue metrics should be interpreted with caution.